# 01 - Data Collection with the SpaceX REST API
**IBM Applied Data Science Capstone - SpaceX Falcon 9 First-Stage Landing Prediction**

Author: Pranav Vyankatesh Jagdale

**Goal:** request past launches from the SpaceX API, resolve the rocket / launchpad / payload / core IDs, and build a Falcon 9 launch table.

> Run every cell in order. Keep your own outputs - they feed your final slides.

In [ ]:
import datetime
from functools import lru_cache
import requests
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Request the launch list

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print("status:", response.status_code)
data = pd.json_normalize(response.json())
data.head(3)

## 2. Keep only the columns we need, single-core / single-payload launches, and the course date cut-off

In [ ]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date'] = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(data.shape)
data.head()

## 3. Helper functions that resolve IDs (cached so each ID is requested once)

In [ ]:
@lru_cache(maxsize=None)
def get_json(endpoint, item_id):
    r = requests.get(f"https://api.spacexdata.com/v4/{endpoint}/{item_id}")
    r.raise_for_status()
    return r.json()

BoosterVersion, PayloadMass, Orbit = [], [], []
LaunchSite, Longitude, Latitude = [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial = [], [], [], []

for _, row in data.iterrows():
    BoosterVersion.append(get_json('rockets', row['rocket'])['name'])
    pad = get_json('launchpads', row['launchpad'])
    LaunchSite.append(pad['name']); Longitude.append(pad['longitude']); Latitude.append(pad['latitude'])
    pl = get_json('payloads', row['payloads'])
    PayloadMass.append(pl['mass_kg']); Orbit.append(pl['orbit'])
    core = row['cores']
    Outcome.append(f"{core['landing_success']} {core['landing_type']}")
    Flights.append(core['flight']); GridFins.append(core['gridfins']); Reused.append(core['reused'])
    Legs.append(core['legs']); LandingPad.append(core['landpad'])
    if core['core'] is not None:
        c = get_json('cores', core['core'])
        Block.append(c['block']); ReusedCount.append(c['reuse_count']); Serial.append(c['serial'])
    else:
        Block.append(None); ReusedCount.append(None); Serial.append(None)
print("lookups done")

## 4. Assemble the dataset

In [ ]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude,
}
df = pd.DataFrame(launch_dict)
df.head()

## 5. Keep Falcon 9 only, fix the flight numbers, and handle missing payload mass

In [ ]:
data_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print(data_falcon9.isnull().sum())
mean_mass = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].replace(np.nan, mean_mass)
print("mean payload mass used:", round(mean_mass, 1))
data_falcon9.to_csv('my_dataset_part_1.csv', index=False)
data_falcon9.shape

## 6. Write your own summary here
- How many launches did you collect?
- Which columns have missing values and what did you do about them?
- Anything that surprised you?